# 📋 CPZÃO · Notebook 01 — Aquisição e Preparação do Dataset

**Auditoria de Calçadas com Visão Computacional** · Detecção de objetos YOLOv8

Este notebook recebe o dataset anotado, **valida a consistência das anotações**, confere o
split train/val/test e gera as **estatísticas que entram no README**.

### Classes

| id | nome | descrição |
|:--:|:--|:--|
| 0 | `buraco` | buraco, cratera, trinca profunda, piso quebrado/faltante |
| 1 | `piso_tatil` | piso tátil direcional ou de alerta |
| 2 | `obstaculo` | poste, lixeira, hidrante, bicicletário, mesa bloqueando a faixa livre |

### Antes de rodar

Você precisa ter concluído o [`CAPTURA.md`](../CAPTURA.md): fotos anotadas no Roboflow e
exportadas em formato **YOLOv8**.

### Ordem de execução

`01_dataset_prep.ipynb` (este) → `02_treino_export.ipynb`

> 🖥️ **Runtime:** `Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)`

## 1. Ambiente

In [ ]:
!pip install -q ultralytics roboflow

import os, sys, glob, shutil, random, json
from pathlib import Path
from collections import Counter, defaultdict

import yaml
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
print("torch   :", torch.__version__)
print("GPU     :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ nenhuma (ative GPU no runtime!)")

## 2. Configuração

Preencha os campos abaixo com os dados do **seu** projeto no Roboflow.

- `ROBOFLOW_API_KEY` → Roboflow → *Settings* → *API Keys*
- `WORKSPACE` / `PROJECT` / `VERSION` → estão no snippet que o Roboflow mostra em
  *Download Dataset → show download code*

In [ ]:
# ─── PREENCHA AQUI ────────────────────────────────────────────────
ROBOFLOW_API_KEY = ""                      # <<< sua API key
WORKSPACE        = ""                      # <<< ex: "fiap-calcadas-xyz"
PROJECT          = "auditoria-calcadas"    # <<< slug do projeto
VERSION          = 1                       # <<< número da versão gerada
# ──────────────────────────────────────────────────────────────────

CLASSES   = ["buraco", "piso_tatil", "obstaculo"]
NUM_CLASS = len(CLASSES)

# imgsz do treino = 192 porque o app mobile (app/src/inference.js) usa INPUT_W/H = 192 fixo.
IMGSZ = 192

print(f"{NUM_CLASS} classes: {CLASSES}")
print(f"imgsz alvo: {IMGSZ} (travado pelo app mobile)")

## 3. Obter o dataset

Rode **a Opção A** (Roboflow, recomendado). Se preferir subir um `.zip` na mão, use a Opção B.

### Opção A — download direto do Roboflow

In [ ]:
from roboflow import Roboflow

rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
ds      = project.version(VERSION).download("yolov8")

DATASET_DIR = Path(ds.location)
print("\n✅ dataset em:", DATASET_DIR)

### Opção B — upload manual de um `.zip`

Use se exportou o dataset como arquivo. Rode a célula, escolha o `.zip` e ele é
descompactado automaticamente.

In [ ]:
# ⚠️ Só rode esta célula se NÃO usou a Opção A.
USAR_UPLOAD_MANUAL = False

if USAR_UPLOAD_MANUAL:
    from google.colab import files
    up = files.upload()
    zip_name = list(up.keys())[0]
    !unzip -q -o "{zip_name}" -d /content/dataset_manual
    DATASET_DIR = Path("/content/dataset_manual")
    print("✅ dataset em:", DATASET_DIR)
else:
    print("pulado (usando Opção A)")

## 4. Normalizar o `data.yaml`

**Esta célula resolve o erro mais comum do pipeline.** O `data.yaml` que o Roboflow gera traz
caminhos relativos (`../train/images`) que quebram quando o Ultralytics muda de diretório de
trabalho — o treino falha com *"Dataset not found"* ou, pior, treina com 0 imagens.

Aqui reescrevemos o arquivo com **caminhos absolutos** e checamos os nomes das classes.

In [ ]:
yaml_path = DATASET_DIR / "data.yaml"
assert yaml_path.exists(), f"data.yaml não encontrado em {DATASET_DIR}"

cfg = yaml.safe_load(yaml_path.read_text())
print("── data.yaml original ──")
print(yaml.dump(cfg, allow_unicode=True))

# Descobre como cada split está nomeado nesta export (valid/ vs val/)
def find_split(*candidates):
    for c in candidates:
        if (DATASET_DIR / c / "images").is_dir():
            return f"{c}/images"
    return None

splits = {
    "train": find_split("train"),
    "val":   find_split("valid", "val"),
    "test":  find_split("test"),
}

novo = {"path": str(DATASET_DIR.resolve()), "nc": NUM_CLASS, "names": CLASSES}
for k, v in splits.items():
    if v:
        novo[k] = v
    else:
        print(f"⚠️  split '{k}' ausente no dataset")

yaml_path.write_text(yaml.dump(novo, allow_unicode=True, sort_keys=False))
DATA_YAML = str(yaml_path)

print("── data.yaml normalizado ──")
print(yaml_path.read_text())

# Checagem de consistência de nomes: a ordem DEVE bater com o treino e com o app
nomes_rf = cfg.get("names", [])
if isinstance(nomes_rf, dict):
    nomes_rf = [nomes_rf[k] for k in sorted(nomes_rf)]
if list(nomes_rf) != CLASSES:
    print("\n🚨 ATENÇÃO: a ordem/nome das classes no Roboflow difere de CLASSES.")
    print(f"   Roboflow : {list(nomes_rf)}")
    print(f"   Esperado : {CLASSES}")
    print("   Os IDs das labels seguem a ordem do ROBOFLOW. Ajuste a lista CLASSES")
    print("   para a ordem do Roboflow e use essa mesma ordem no app (CLASS_NAMES).")
else:
    print("\n✅ nomes e ordem das classes conferem")

## 5. Validação das anotações

Varre todos os arquivos `.txt` procurando os problemas que silenciosamente arruínam o treino:

| problema | efeito |
|:--|:--|
| `class_id` fora de `[0, nc-1]` | o treino aborta com *IndexError* |
| coordenada fora de `[0,1]` | caixa cortada ou ignorada |
| largura/altura = 0 | caixa degenerada, vira `NaN` na loss |
| imagem sem label correspondente | tratada como "fundo puro" — derruba o recall |
| label sem imagem | arquivo órfão, ignorado |

In [ ]:
def img_dir(split_rel):  return DATASET_DIR / split_rel
def lbl_dir(split_rel):  return DATASET_DIR / split_rel.replace("/images", "/labels")

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
problemas = []
stats     = {}

for split, rel in splits.items():
    if not rel:
        continue

    imgs = sorted([p for p in img_dir(rel).iterdir() if p.suffix.lower() in IMG_EXT])
    lbls = sorted(lbl_dir(rel).glob("*.txt"))

    cont_cls   = Counter()
    n_caixas   = 0
    sem_label  = []
    vazios     = []
    areas      = []

    for ip in imgs:
        lp = lbl_dir(rel) / (ip.stem + ".txt")
        if not lp.exists():
            sem_label.append(ip.name)
            continue

        linhas = [l for l in lp.read_text().strip().splitlines() if l.strip()]
        if not linhas:
            vazios.append(ip.name)
            continue

        for n, linha in enumerate(linhas, 1):
            partes = linha.split()
            if len(partes) < 5:
                problemas.append(f"[{split}] {lp.name}:{n} — formato inválido: '{linha}'")
                continue

            cid = int(float(partes[0]))
            x, y, w, h = map(float, partes[1:5])

            if not (0 <= cid < NUM_CLASS):
                problemas.append(f"[{split}] {lp.name}:{n} — class_id {cid} fora de [0,{NUM_CLASS-1}]")
            if not all(0 <= v <= 1 for v in (x, y, w, h)):
                problemas.append(f"[{split}] {lp.name}:{n} — coords fora de [0,1]: {x:.3f} {y:.3f} {w:.3f} {h:.3f}")
            if w <= 0 or h <= 0:
                problemas.append(f"[{split}] {lp.name}:{n} — caixa degenerada (w={w}, h={h})")

            cont_cls[cid] += 1
            n_caixas += 1
            areas.append(w * h)

    orfaos = [l.stem for l in lbls if not any((img_dir(rel) / (l.stem + e)).exists() for e in IMG_EXT)]

    stats[split] = {
        "imagens":        len(imgs),
        "caixas":         n_caixas,
        "por_classe":     {CLASSES[i]: cont_cls.get(i, 0) for i in range(NUM_CLASS)},
        "sem_label":      sem_label,
        "labels_vazios":  vazios,
        "labels_orfaos":  orfaos,
        "areas":          areas,
    }

print("═" * 62)
for split, s in stats.items():
    print(f"\n▸ {split.upper():5s}  {s['imagens']:4d} imagens   {s['caixas']:4d} caixas")
    for c, n in s["por_classe"].items():
        print(f"      {c:<12s} {n:4d}")
    if s["sem_label"]:
        print(f"      ⚠️  {len(s['sem_label'])} imagem(ns) sem label: {s['sem_label'][:4]}")
    if s["labels_vazios"]:
        print(f"      ℹ️  {len(s['labels_vazios'])} label(s) vazio(s) (negativos — ok em pequena quantidade)")
    if s["labels_orfaos"]:
        print(f"      ⚠️  {len(s['labels_orfaos'])} label(s) órfão(s): {s['labels_orfaos'][:4]}")

print("\n" + "═" * 62)
if problemas:
    print(f"🚨 {len(problemas)} problema(s) encontrado(s):\n")
    for p in problemas[:30]:
        print("  •", p)
    if len(problemas) > 30:
        print(f"  ... e mais {len(problemas) - 30}")
    print("\n→ Corrija no Roboflow, gere uma nova versão e rode este notebook de novo.")
else:
    print("✅ nenhum problema estrutural nas anotações")

## 6. Conformidade com a rubrica

Checagem automática dos requisitos que valem os **4,0 pontos** de *Dataset & Rotulagem*.

In [ ]:
total_img   = sum(s["imagens"] for s in stats.values())
total_cx    = sum(s["caixas"]  for s in stats.values())
por_classe  = {c: sum(s["por_classe"][c] for s in stats.values()) for c in CLASSES}

checks = [
    ("≥ 60 imagens rotuladas",        total_img >= 60,                      f"{total_img} imagens"),
    ("≥ 3 classes distintas",         NUM_CLASS >= 3,                       f"{NUM_CLASS} classes"),
    ("todas as classes com instância", all(v > 0 for v in por_classe.values()), str(por_classe)),
    ("≥ 20 instâncias por classe",    all(v >= 20 for v in por_classe.values()), str(por_classe)),
    ("split train presente",          stats.get("train", {}).get("imagens", 0) > 0, f"{stats.get('train',{}).get('imagens',0)} img"),
    ("split val presente",            stats.get("val", {}).get("imagens", 0)   > 0, f"{stats.get('val',{}).get('imagens',0)} img"),
    ("split test presente",           stats.get("test", {}).get("imagens", 0)  > 0, f"{stats.get('test',{}).get('imagens',0)} img"),
    ("sem erros estruturais",         len(problemas) == 0,                   f"{len(problemas)} erros"),
]

print("═" * 62)
print("CONFORMIDADE COM A RUBRICA")
print("═" * 62)
for nome, ok, detalhe in checks:
    print(f"  {'✅' if ok else '❌'}  {nome:<32s} {detalhe}")
print("═" * 62)

if all(ok for _, ok, _ in checks):
    print("🎉 dataset aprovado — siga para o notebook 02")
else:
    print("⚠️  resolva os itens ❌ antes de treinar")

## 7. Estatísticas e gráficos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

# (a) instâncias por classe, empilhado por split
ordem   = [s for s in ["train", "val", "test"] if s in stats]
largura = 0.6
base    = np.zeros(NUM_CLASS)
cores   = {"train": "#2563eb", "val": "#f59e0b", "test": "#10b981"}

for sp in ordem:
    vals = np.array([stats[sp]["por_classe"][c] for c in CLASSES], dtype=float)
    axes[0].bar(CLASSES, vals, largura, bottom=base, label=sp, color=cores[sp])
    base += vals

for i, c in enumerate(CLASSES):
    axes[0].text(i, base[i] + max(base) * 0.02, int(base[i]), ha="center", fontweight="bold")

axes[0].set_title("Instâncias por classe", fontweight="bold")
axes[0].set_ylabel("nº de caixas")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=15)

# (b) imagens por split
vals = [stats[sp]["imagens"] for sp in ordem]
bars = axes[1].bar(ordem, vals, color=[cores[s] for s in ordem])
axes[1].bar_label(bars, fmt="%d", fontweight="bold")
axes[1].set_title(f"Imagens por split (total {total_img})", fontweight="bold")
axes[1].set_ylabel("nº de imagens")

# (c) distribuição do tamanho das caixas
todas_areas = np.concatenate([stats[sp]["areas"] for sp in ordem if stats[sp]["areas"]])
axes[2].hist(np.sqrt(todas_areas) * 100, bins=28, color="#6366f1", edgecolor="white")
axes[2].set_title("Tamanho das caixas", fontweight="bold")
axes[2].set_xlabel("√área (% do lado da imagem)")
axes[2].set_ylabel("frequência")
axes[2].axvline(np.sqrt(todas_areas).mean() * 100, color="#dc2626", ls="--",
                label=f"média {np.sqrt(todas_areas).mean()*100:.0f}%")
axes[2].legend()

plt.tight_layout()
plt.savefig("dataset_stats.png", dpi=130, bbox_inches="tight")
plt.show()

print(f"\nmédia de caixas por imagem: {total_cx/total_img:.2f}")
print(f"balanceamento (menor/maior classe): {min(por_classe.values())/max(por_classe.values()):.2f}")
print("  → acima de 0.5 é saudável; muito abaixo, capture mais fotos da classe minoritária")

## 8. Inspeção visual das anotações

Caixas desenhadas sobre as imagens de treino. **Olhe de verdade para esta grade** — é o jeito
mais rápido de pegar Auto-Orient desligado (caixas giradas 90°) ou classes trocadas.

In [ ]:
from PIL import ImageDraw

PALETA = {0: "#ff1744", 1: "#ffd600", 2: "#00e676"}  # mesmas cores do app

rel   = splits["train"]
amostra = sorted([p for p in img_dir(rel).iterdir() if p.suffix.lower() in IMG_EXT])
random.seed(42)
amostra = random.sample(amostra, min(9, len(amostra)))

fig, axes = plt.subplots(3, 3, figsize=(14, 14))
for ax, ip in zip(axes.ravel(), amostra):
    im = Image.open(ip).convert("RGB")
    W, H = im.size
    dr = ImageDraw.Draw(im)

    lp = lbl_dir(rel) / (ip.stem + ".txt")
    if lp.exists():
        for linha in lp.read_text().strip().splitlines():
            if not linha.strip():
                continue
            cid, x, y, w, h = linha.split()[:5]
            cid = int(float(cid)); x, y, w, h = map(float, (x, y, w, h))
            x1, y1 = (x - w/2) * W, (y - h/2) * H
            x2, y2 = (x + w/2) * W, (y + h/2) * H
            cor = PALETA.get(cid, "#ffffff")
            dr.rectangle([x1, y1, x2, y2], outline=cor, width=4)
            dr.rectangle([x1, y1 - 20, x1 + 9 * len(CLASSES[cid]), y1], fill=cor)
            dr.text((x1 + 3, y1 - 18), CLASSES[cid], fill="black")

    ax.imshow(im); ax.axis("off"); ax.set_title(ip.name[:26], fontsize=8)

for ax in axes.ravel()[len(amostra):]:
    ax.axis("off")

plt.tight_layout()
plt.savefig("anotacoes_amostra.png", dpi=110, bbox_inches="tight")
plt.show()

## 9. Exportar estatísticas para o README

Gera a tabela em Markdown pronta para colar na seção *Estatísticas do Dataset* do `README.md`.

In [ ]:
linhas = []
linhas.append("| split | imagens | caixas | " + " | ".join(f"`{c}`" for c in CLASSES) + " |")
linhas.append("|:--|--:|--:|" + "--:|" * NUM_CLASS)
for sp in ordem:
    s = stats[sp]
    linhas.append(f"| **{sp}** | {s['imagens']} | {s['caixas']} | "
                  + " | ".join(str(s["por_classe"][c]) for c in CLASSES) + " |")
linhas.append(f"| **TOTAL** | **{total_img}** | **{total_cx}** | "
              + " | ".join(f"**{por_classe[c]}**" for c in CLASSES) + " |")

tabela = "\n".join(linhas)

resumo = f'''## 📊 Estatísticas do Dataset

{tabela}

- **Total de imagens rotuladas:** {total_img}
- **Total de caixas anotadas:** {total_cx}
- **Média de caixas por imagem:** {total_cx/total_img:.2f}
- **Divisão:** {stats.get("train",{}).get("imagens",0)} train / {stats.get("val",{}).get("imagens",0)} val / {stats.get("test",{}).get("imagens",0)} test
- **Balanceamento entre classes:** {min(por_classe.values())/max(por_classe.values()):.2f} (menor/maior)
- **Anotação:** Roboflow · formato YOLOv8 · Auto-Orient + Resize *fit* 640
- **Augmentation (só no train):** flip horizontal, brilho ±20%, blur ≤1.5px, rotação ±10°
'''

print(resumo)
with open("dataset_stats.md", "w") as f:
    f.write(resumo)

json.dump(
    {"total_imagens": total_img, "total_caixas": total_cx, "por_classe": por_classe,
     "por_split": {sp: {"imagens": stats[sp]["imagens"], "caixas": stats[sp]["caixas"],
                        "por_classe": stats[sp]["por_classe"]} for sp in ordem},
     "classes": CLASSES, "imgsz": IMGSZ},
    open("dataset_stats.json", "w"), indent=2, ensure_ascii=False)

print("\n✅ gerados: dataset_stats.md · dataset_stats.json · dataset_stats.png · anotacoes_amostra.png")

In [ ]:
# Baixa os artefatos para commitar em docs/
from google.colab import files
for f in ["dataset_stats.md", "dataset_stats.json", "dataset_stats.png", "anotacoes_amostra.png"]:
    try:
        files.download(f)
    except Exception as e:
        print("não baixou", f, e)

## 10. Guardar o caminho do dataset

O notebook 02 precisa deste caminho. Copie a saída abaixo.

In [ ]:
print("DATA_YAML =", repr(DATA_YAML))
print("DATASET_DIR =", repr(str(DATASET_DIR)))

# Se estiver rodando os dois notebooks na MESMA sessão do Colab, isto já persiste:
%store DATA_YAML
%store DATASET_DIR
%store CLASSES
%store IMGSZ

---

## ✅ Próximo passo

Dataset validado e aprovado na rubrica → abra **[`02_treino_export.ipynb`](02_treino_export.ipynb)**.

Commite em `docs/metricas/` os arquivos baixados:
`dataset_stats.png` e `anotacoes_amostra.png` são evidência direta de
*"rotulagem com consistência"* na correção.